# Chapter 02 — Install, First Session, First Task

**Companion to *Pi Agents*** · [`Pi Agents` chapter 02](https://programmer.ie/books/pi/02-chapter/)

| | |
|---|---|
| Chapter | `02-chapter.md` · weight 2 · `/books/pi/02-chapter/` |
| Notebook | `notebooks/pi/02-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

The working directory is the identity key. **What does it decide** —
and can you see it decide the same thing twice, from two runs?

The chapter opens with installing Pi, logging in, and running a first task. The
install is the least interesting part and the one you cannot check without a
network. The identity key is the part you *can*: it decides what Pi discovers
underneath you and which saved conversation `--continue` resumes.


## What this notebook establishes

- The bundled `pi` executable in `examples/node_modules/` reports the pinned version. This is the same file `npm exec pi` runs, resolved through the package's own `bin` mapping.
- A session is written to a directory whose name is derived from the working directory: the leading separator is dropped and `/`, `\` and `:` become `-`.
- Two runs in two different working directories, sharing one agent directory, land in **two different session groups** — so `--continue` cannot cross between them.

## What this notebook does **not** establish

- **Nothing about installing Pi for a reader.** No npm registry, no `npm install -g`, no version-check service. This notebook exercises the pinned package already in the repository.
- **Nothing about logging in.** There is no credential and no OAuth flow here; chapter 3 is where a credential's source matters.
- **Nothing about task phrasing.** The chapter contrasts *"Fix the bug."* with a scoped task, and `metadata/10-chapter.yaml` records that the effect of phrasing on completion rate **is not measured**. It cannot be measured with a scripted model, and it is not measured here.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the shipped `pi` binary, offline, with the book's scripted provider.
* **Evidence scope:** `shipped_binary`. This chapter has no example directory of its own; it runs against the bundled executable through `examples/harness/cli.ts`.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(2))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/02-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: which executable, and which version

`harness/cli.ts` resolves the binary through the pinned package's own `bin`
mapping rather than hard-coding a path into `dist/`. That matters: `dist/cli.js`
is an internal module the bundle happens to contain, so running it would test a
different program from the one you install.

In [3]:
ABOUT = pinb.driver_source("ch02-version.ts")
about = pinb.driver(ABOUT, label="ch02-version", timeout=120)
print(pinb.table(["property", "value"], [[k, v] for k, v in about.items()], indent=""))
print()
assert about["exit"] == 0 and about["reportedVersion"] == pinb.PIN, (
    f"the bundled binary reported {about['reportedVersion']!r}, not the pin {pinb.PIN}")
print(f"the shipped executable in this repository is Pi {about['reportedVersion']}")

property  value
---------------  ------------------
binMapping  dist/bundle/cli.js
exit  0
reportedVersion  1.0.4
helpIsAvailable  True

the shipped executable in this repository is Pi 1.0.4


## Experiment: the working directory as identity key

Two runs, same agent directory, **different working directories**. The
question is not whether both succeed — it is whether the session files land in
different places, and what that makes `--continue` mean.

The run below is offline (`PI_OFFLINE=1`), writes nothing to your own Pi
configuration (`PI_CODING_AGENT_DIR` points at a fresh temporary directory) and
uses the book's scripted provider.

In [4]:
IDENTITY = pinb.driver_source("ch02-identity.ts")
identity = pinb.driver(IDENTITY, label="ch02-identity", timeout=240)
for r in identity["runs"]:
    print(f"run in cwd \"{r['cwdShape']}\": exit={r['exit']}, stdout={r['stdout']!r}")
    print(f"  session groups afterwards: {r['groupNames']}")

print()
print(pinb.table(["session group", "file"], [[f["group"], f["file"]] for f in identity["sessionFiles"]], indent=""))

run in cwd "alpha": exit=0, stdout='ok from alpha'
  session groups afterwards: ['--<tmp-root>-pinb-ch02-alpha-qsavbu--']
run in cwd "beta": exit=0, stdout='ok from beta'
  session groups afterwards: ['--<tmp-root>-pinb-ch02-alpha-qsavbu--', '--<tmp-root>-pinb-ch02-beta-njI1Kn--']

session group  file
-------------------------------------  -------------------------------------------------------------------
--<tmp-root>-pinb-ch02-alpha-qsavbu--  2026-10-07T21-26-39-085Z_01a11842-e2ec-7317-bd16-457ede38ce74.jsonl
--<tmp-root>-pinb-ch02-beta-njI1Kn--  2026-10-07T21-26-39-372Z_01a11842-e40b-752f-8ba4-c6dd9485c6d9.jsonl


Both runs printed their answer and exited 0. The interesting row is the
group directory name: it is the working directory with its separators flattened,
and the two runs produced **two** of them.

That is the whole answer to "which conversation does `--continue` resume?" —
it resumes the one belonging to *this directory*, and there is no global "last
session" to fall back on.

In [5]:
groups = sorted({g["group"] for g in identity["sessionFiles"]})
names = sorted({f["file"] for f in identity["sessionFiles"]})
filename_pattern = __import__("re").compile(r"^\d{4}-\d\d-\d\dT[\d:-]+Z_[0-9a-f-]{36}\.jsonl$")

pinb.show_checks([
    pinb.check("both runs succeeded", all(r["exit"] == 0 for r in identity["runs"]),
               str([r["exit"] for r in identity["runs"]])),
    pinb.check("each run produced its own session group", len(groups) == 2, str(len(groups)) + " groups"),
    pinb.check("no group name contains a path separator or a colon",
               not any(set(g) & set("/\\:") for g in groups), "separators flattened"),
    pinb.check("every session file name matches the documented shape",
               all(filename_pattern.match(n) for n in names), "ISO timestamp + UUID"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  --------------------
PASS  both runs succeeded  [0, 0]
PASS  each run produced its own session group  2 groups
PASS  no group name contains a path separator or a colon  separators flattened
PASS  every session file name matches the documented shape  ISO timestamp + UUID

4/4 assertions held.


## Contrasting case: the rule is not injective

The group names above are derived from temporary paths that differ by more than
their label. Strip both back to the label and the rule is visible:
`^[/\\]` removed, then every `/`, `\` and `:` replaced with `-`, wrapped in `--`.

A collision is therefore *possible*: `a/b` and `a-b` flatten to the same string.
The book states the rule; it does not claim the rule is injective. The cell below
checks that, and it is the reason a session directory name is not something to
parse.

## Your turn: predict, then run

**Predict first.** If two temporary working directories differed only by one
character — `a-b` and `a/b` — how many session groups would there be?

**Then run it.** The cell applies the documented flattening rule to the two names.

In [6]:
LABELS = ["a-b", "a/b"]


def flatten(path: str) -> str:
    """The documented session-group rule, restated so the collision is visible.

    The authority for the rule is the session file written above; this only
    applies it to names the notebook chose.
    """
    import re

    body = re.sub(r"^[/\\]", "", path.replace("\\", "/"))
    return "--" + re.sub(r"[/\\:]", "-", body) + "--"


flat = [flatten(f"/tmp/proj/{x}") for x in LABELS]
print("paths         :", [f"/tmp/proj/{x}" for x in LABELS])
print("flattened     :", flat)
print("distinct      :", sorted(set(flat)))
print("predicted groups:", len(set(flat)))
print()
assert len(set(flat)) == 1, "the rule is not injective: two paths, one session group"
print("held: the rule is documented, injectivity is not. Two paths can share a group.")

paths         : ['/tmp/proj/a-b', '/tmp/proj/a/b']
flattened     : ['--tmp-proj-a-b--', '--tmp-proj-a-b--']
distinct      : ['--tmp-proj-a-b--']
predicted groups: 1

held: the rule is documented, injectivity is not. Two paths can share a group.


## Interpretation

The practical decision this chapter supports: **treat the directory as part of
the task's identity**. Two repositories are two conversations, and switching
directories mid-task means starting a new session rather than continuing the old
one. That is also why chapter 8's project trust, chapter 7's settings layering
and chapter 28's context-file discovery are all keyed on the same directory — the
identity key is chosen before any of them run.

### What this chapter does not settle

* The install itself. Three installers exist (npm, a managed installer, a nix
  profile since 1.0.1); choosing between them is a packaging decision this
  notebook has no opinion about.
* Task shape. *"Fix the bug."* and a scoped task differ, but no measurement in
  this book establishes by how much, and the faux provider could not produce one.
* Native Windows and Termux setup, which `metadata/02-chapter.yaml` records as
  outside the chapter's scope.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the bundled `pi` executable and two offline runs through `examples/harness/cli.ts`.

**Evidence scope:** `shipped_binary`. This chapter has no example directory, so there are no ledger rows for it; the evidence here is this notebook's own execution, which is reported as such.

### What was read or run

- **Ran** `pi --version` and `pi --help` on the binary resolved through the pinned package's `bin` mapping (`shipped_binary`).
- **Ran** two scripted `--print` runs with `session: true`, in two fresh working directories, sharing one fresh `PI_CODING_AGENT_DIR`, with `PI_OFFLINE=1`.
- **Read** the session-file naming rule in chapter 5's example assertions and the harness source that implements it.

### Limitations

- **The temporary paths are machine-shaped.** The flattening rule is the chapter's; the specific group names depend on where the OS put the temporary directory. That is why only the *count* and the *character set* are asserted, not the names.
- **No session was resumed.** Two runs in two directories show that sessions are kept apart. Showing that `--continue` picks the right one needs a third run in a directory that already has a session, which is left as the reader's exercise rather than asserted here.
- **The model is scripted.** Every run answered `ok from <label>` because the faux provider said so. Nothing here is evidence about model behaviour, and nothing about a real task.

### Optional work that was NOT run

- **Installing Pi** from npm, the managed installer, or the nix profile. Documented in the chapter; not run here.
- **Logging in** via `/login` or an OAuth flow. No credential is used anywhere in this set.
- **A `--continue` invocation** in a directory that already holds a session.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
